# 🪐 PROJE 1 — Uzay: Yeni Bir Dünya Buldun mu?
## Mini Proje — Araştırma Projeniz
*Bugün (Gün 2) seçtiğiniz proje — kampın son günü "Bilimsel Yöntem" oturumunda sunulacak*

**Veri seti:** ~7300 GERÇEK NASA Kepler ötegezegen adayı
**Kaynak:** NASA Exoplanet Archive — gerçek 2009-2018 Kepler uydusu gözlemleri

Astronomlar yıldız parlaklığındaki kısa düşüşlere bakar (transit). Bazıları gerçek gezegen, bazıları ikili yıldız sistemi veya gürültü.

### 🎯 Sizin göreviniz: Hangileri **gerçek**, hangileri **yanlış pozitif**?

**Aşağıdaki seçeneklerden en az 3'ünü deneyin:**
1. **Sınıflandırma modeli** eğitin (gerçek vs yanlış pozitif)
2. 🌟 **Yıldız tipi analizi:** M cüce vs G yıldızı — gezegenler farklı mı?
3. 🔥 **Sıcak Jüpiter avı:** Çok kısa yörünge + büyük gezegen — kaç tane?
4. 🌍 **Yaşanabilir bölge** Goldilocks gezegenleri
5. 🔭 **Çok-gezegenli sistemler:** Tek yıldızın kaç gezegeni var?
6. 🌎 **Dünya benzeri** filtreleyin (0.8-1.5 yarıçap, 250-350 K)
7. 😨 Modelin **en güvensiz** olduğu adaylar — astronomlara "şuna tekrar bakın" listesi

---

🌌 **İlham:** NASA Kepler 2009-2018 arası 5000+ gerçek ötegezegen keşfetti. Sizin modeliniz aynı işin bir prototipi!

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/STEM_yaz_kampi_2026')
from python_code.helper_tr import *
import pandas as pd
import numpy as np

uzay = veri_yukle('input_data/uzay_otegezegen.csv')
print(f'\nGerçek gezegen oranı: %{uzay["gercek_gezegen"].mean()*100:.1f}')
print(f'Yaşanabilir bölge oranı: %{uzay["yasanabilir_bolge"].mean()*100:.1f}')

---
### 👥 Grup Tartışması — Başlamadan Önce
*Grupça 2–3 dakika konuşun; tahminlerinizi sonra veriyle karşılaştıracaksınız.*
- Bir teleskop, bir yıldızın önünden geçen bir gölge gördü. Bunun **gerçek bir gezegen** mi yoksa **yanlış alarm** mı olduğunu anlamak için hangi ipuçlarına bakardınız? En az 3 fikir üretin.
- Sizce evrende kaç gezegen olabilir? Neye dayanarak tahmin ediyorsunuz?

## Adım 1: Veriyi Tanı

Sütunlar:
- `aday_id`: Gerçek KOI numarası (K00752 = Kepler-227 sistemi gibi)
- `yorunge_periyodu_gun`, `transit_suresi_saat`, `transit_derinligi_ppm`
- `gezegen_yaricap_dunya`, `denge_sicakligi_K`
- `yildiz_sicakligi_K`, `yildiz_yaricap_gunes`
- `sinyal_gurultu_orani`

In [ ]:
veriyi_ozetle(uzay)

## 🌟 Adım 2: Yıldız Tipi Analizi (M cüce vs G yıldız)

Yıldızlar sıcaklıklarına göre sınıflandırılır:
- **M cüce** (<3700 K): Soğuk, küçük, kırmızı. Yaşanabilir bölge yıldıza yakın
- **K** (3700-5200 K): Turuncu cüce
- **G** (5200-6000 K): Güneş gibi sarı
- **F** (6000-7500 K): Beyaz-sarı
- **A** (>7500 K): Beyaz, sıcak, kısa ömürlü

In [ ]:
def yildiz_tipi(t):
    if t < 3700: return 'M (kırmızı cüce)'
    if t < 5200: return 'K (turuncu)'
    if t < 6000: return 'G (güneş gibi)'
    if t < 7500: return 'F (beyaz-sarı)'
    return 'A (beyaz)'
uzay['yildiz_tipi'] = uzay['yildiz_sicakligi_K'].apply(yildiz_tipi)

# Yıldız tipine göre gerçek gezegen oranı
tip_analiz = uzay.groupby('yildiz_tipi').agg(
    aday_sayisi=('gercek_gezegen', 'size'),
    gercek_oran=('gercek_gezegen', 'mean'),
    ort_yaricap=('gezegen_yaricap_dunya', 'mean'),
    ort_periyot=('yorunge_periyodu_gun', 'mean'),
).round(2)
print('Yıldız tipine göre:')
print(tip_analiz)

In [ ]:
cubuk_grafigi(uzay, kategori_sutunu='yildiz_tipi',
              baslik='Yıldız Tipine Göre Aday Sayısı (Kepler nereye baktı?)')

**Görev:** Kepler hangi yıldız tipine en çok baktı? G ve K niye baskın? (İpucu: Kepler bilinçli olarak Güneş benzeri yıldızlara odaklandı)

---
### 👥 Grup Tartışması
- Kepler çoğunlukla belli tip yıldızlara baktı. Veride onların çok olması **'evrende en çok onlar var'** mı demek, yoksa **'biz en çok onlara baktık'** mı? Bu fark neden önemli?
- Buna bilimde **seçim yanlılığı** (selection bias) denir. Günlük hayattan benzer bir örnek bulabilir misiniz?

## 🔥 Adım 3: Sıcak Jüpiter Avı

**Sıcak Jüpiter** = Yörüngesi <10 gün + yarıçapı >8 Dünya = yıldızına çok yakın dev gezegen.
Bu gezegenler astronomları uzun yıllar şaşırttı (Jüpiter'imiz 12 yıllık yörüngede!).

In [ ]:
sicak_jupiter = uzay[
    (uzay['yorunge_periyodu_gun'] < 10) &
    (uzay['gezegen_yaricap_dunya'] > 8) &
    uzay['gercek_gezegen']
]
print(f'🔥 Sıcak Jüpiter sayısı: {len(sicak_jupiter)}')
print(f'   Ortalama denge sıcaklığı: {sicak_jupiter["denge_sicakligi_K"].mean():.0f} K')
print(f'   En sıcak: {sicak_jupiter["denge_sicakligi_K"].max():.0f} K (kurşun erir!)')

# Dağılım grafiği
dagilim_grafigi(
    uzay[uzay['gercek_gezegen']],
    x_sutunu='yorunge_periyodu_gun', y_sutunu='gezegen_yaricap_dunya',
    baslik='Gerçek Gezegenler — Yörünge vs Yarıçap (Sıcak Jüpiterler sol-üst köşe)',
)

## Adım 4: Sınıflandırma Modeli — Gerçek mi Yanlış Pozitif mi?

In [ ]:
ozellikler = ['yorunge_periyodu_gun', 'transit_suresi_saat', 'transit_derinligi_ppm',
              'gezegen_yaricap_dunya', 'yildiz_sicakligi_K', 'yildiz_yaricap_gunes',
              'denge_sicakligi_K', 'sinyal_gurultu_orani']
uzay_clean = uzay.dropna(subset=ozellikler)
X = uzay_clean[ozellikler]
y = uzay_clean['gercek_gezegen']

X_egitim, X_test, y_egitim, y_test = egitim_test_bol(X, y)
model = rastgele_orman_egit(X_egitim, y_egitim)
dogruluk = model_degerlendir(model, X_test, y_test, sinif_adlari=['Yanlış Pozitif', 'Gerçek Gezegen'])

In [ ]:
oznitelik_onemi_grafigi(model, oznitelik_adlari=ozellikler, en_iyi_n=8,
                         baslik='Gerçek Gezegeni Ele Veren İpuçları')

**Görev:** En önemli 3 öznitelik hangisi? Astronom mantığıyla anlamlı mı? (İpucu: gerçek gezegenler genelde Dünya benzeri küçük; ikili yıldız sistemi büyük "transitler" üretir)

---
### 👥 Grup Tartışması
- Modelin en çok güvendiği ipucu sizce **mantıklı** mı? Bir astronom bu modele güvenir mi?
- Model yanlışlıkla 'gezegen' dediği bir sinyalle koca bir 'keşif' duyurulsa ne olur? Bilimde bir sonucu açıklamadan önce neden tekrar tekrar kontrol ederiz?

## 🌍 Adım 5: Yaşanabilir Bölge Avı

In [ ]:
gercek_gezegenler = uzay_clean[uzay_clean['gercek_gezegen']]
yasanabilir = gercek_gezegenler[gercek_gezegenler['yasanabilir_bolge']]
print(f'Gerçek gezegen sayısı: {len(gercek_gezegenler)}')
print(f'Yaşanabilir bölgede: {len(yasanabilir)}')

# Yaşanabilir gezegenlerin yıldız tipi dağılımı
print(f'\nYaşanabilir gezegenlerin yıldız tipi:')
print(yasanabilir.groupby('yildiz_tipi').size())

In [ ]:
# Dünya benzeri filtre
dunya_benzeri = yasanabilir[
    (yasanabilir['gezegen_yaricap_dunya'] >= 0.8) &
    (yasanabilir['gezegen_yaricap_dunya'] <= 1.5) &
    (yasanabilir['denge_sicakligi_K'] >= 250) &
    (yasanabilir['denge_sicakligi_K'] <= 350)
]
print(f'🌎 Dünya benzeri sayısı: {len(dunya_benzeri)}')
if len(dunya_benzeri) > 0:
    print('\nİlk 10 Dünya benzeri:')
    print(dunya_benzeri[['aday_id', 'gezegen_yaricap_dunya', 'denge_sicakligi_K',
                          'yorunge_periyodu_gun', 'yildiz_tipi']].head(10).to_string(index=False))

## 🔭 Adım 6 (Seçenek): Çok-Gezegenli Sistemler

Bazı yıldızların birden fazla gezegeni var. KOI numarası `K00752.01`, `K00752.02` aynı yıldızın 1. ve 2. gezegeni demek!

In [ ]:
# Yıldız ID'sini ayır (K00752.01 → K00752)
uzay_clean['yildiz_id'] = uzay_clean['aday_id'].str.split('.').str[0]
yildiz_gezegen = uzay_clean[uzay_clean['gercek_gezegen']].groupby('yildiz_id').size()
print('Yıldız başına gerçek gezegen sayısı dağılımı:')
print(yildiz_gezegen.value_counts().sort_index())

cok_gezegen = yildiz_gezegen[yildiz_gezegen >= 3]
print(f'\n🔭 3+ gezegenli sistem: {len(cok_gezegen)}')
print(f'En çok gezegenli sistem: {yildiz_gezegen.idxmax()} → {yildiz_gezegen.max()} gezegen')

**Görev:** En çok gezegenli sistem kaç gezegen barındırıyor? Güneş sistemimizle karşılaştırın (8 gezegen).

## 😨 Adım 7: Modelin En Güvensiz Olduğu Adaylar

In [ ]:
olasiliklar = model.predict_proba(X_test)
guvensizlik = np.abs(olasiliklar[:, 1] - 0.5)
test_df = X_test.copy()
test_df['gercek'] = y_test.values
test_df['olasilik'] = olasiliklar[:, 1]
test_df['guvensizlik'] = guvensizlik

en_guvensiz = test_df.sort_values('guvensizlik').head(10)
print('🤔 Modelin en güvensiz olduğu 10 aday (NASA için "tekrar bak" listesi):')
print(en_guvensiz[['gezegen_yaricap_dunya', 'transit_derinligi_ppm',
                    'sinyal_gurultu_orani', 'olasilik', 'gercek']].to_string())

---
### 👥 Grup Tartışması — Sunuma Hazırlık
- Bugün bulduğunuz **en şaşırtıcı** şey neydi? Grupça tek bir 'baş bulgu' seçin.
- Sizi dinleyen biri 'ee, ne olmuş?' derse nasıl cevap verirsiniz? Bulgunuz neden önemli?

## Adım 8: Sunum için notlarınız

- Toplam aday sayısı: ___
- Gerçek gezegen oranı: %___
- 🌟 En çok aday hangi yıldız tipinde: ___
- 🔥 Sıcak Jüpiter sayısı: ___ (en sıcak ___ K)
- 🌍 Yaşanabilir bölgedeki gerçek gezegen: ___
- 🌎 Dünya benzeri sayısı: ___
- 🔭 En çok gezegenli sistem: ___ gezegen
- Modelin doğruluğu: %___
- En önemli 3 öznitelik:

  1. ___________
  2. ___________
  3. ___________
- **Şaşırtıcı bulgu:** _________________________
- **Kariyer:** NASA'da ML mühendisi olmak ister misiniz? TÜBİTAK Uzay Araştırmaları Enstitüsü Türkiye'de var!

🚀 **Sunum ipucu:** Slaytınıza NASA görseli ekleyin (kamuya açık, ücretsiz!).

📝 Bilimsel yöntem şablonunu doldurun!

---
## 🚀 Hızlı Bitirenler İçin BONUS

1. **Süper-Dünyalar:** Yarıçap 1.5-2.5 Dünya arası — kaç tane?
2. **Mini-Neptünler:** Yarıçap 2-4 Dünya — "ara cins" gezegenler, Güneş sistemimizde yok!
3. **TRAPPIST-1 sistemi:** 7 Dünya benzeri gezegen tek yıldız etrafında. Bu veride benzer sistem var mı?
4. **Habitabilite haritası:** Yıldız sıcaklığı vs gezegen-yıldız uzaklığı, yaşanabilir bölgeyi gösteren grafik
5. **Kümeleme:** `KMeans` ile 4 kümeye ayır — kayalı, gaz, sıcak, soğuk gibi grupları görebiliyor musunuz?

## 🆘 Yardım — Sık Karşılaşılan Hatalar

**❌ `KeyError: 'gercek_gezegen'`** → Boolean sütun. `y = uzay['gercek_gezegen']`

**❌ Doğruluk %100** → Veri sızıntısı. `yasanabilir_bolge` veya `durum` X'te olmamalı

**❌ Dünya benzeri = 0** → Bu doğru olabilir! Gerçek hayatta da Dünya benzeri çok azdır. Filtreleri biraz gevşetin

**❌ NaN bol** → `denge_sicakligi_K` bazılarında eksik. `dropna(subset=ozellikler)` kullanın

**❌ Yıldız tipi sütunu çıkmadı** → `apply` fonksiyonunu çağırdınız mı? `uzay['yildiz_tipi'] = uzay['yildiz_sicakligi_K'].apply(yildiz_tipi)`

---
# 🧑‍🏫 Öğretmen Rehberi & Cevap Anahtarı
*Bu bölüm öğretmen içindir: beklenen sonuçlar, sık takılma noktaları ve tartışma soruları. Öğrenciler projeyi yaptıktan sonra bakabilir.*

# 🎓 PROJE 1 EĞİTMEN — Uzay (Ötegezegen Tespiti)
## Mini Proje Saha Rehberi

**Bu notebook eğitmenler içindir.**

## ⚠️ Veri Hakkında

Bu **GERÇEK NASA verisidir** — NASA Exoplanet Archive (exoplanetarchive.ipac.caltech.edu) TAP API'sinden çekildi. KOI numaraları gerçek (K00752, K00754, vb.). Sınıfta açıklayın:

> "Bu veri **gerçek NASA Kepler verisidir**. K00752 = Kepler-227 sistemi. Bu ötegezegenler 2009-2018 arası gerçekten gözlemlendi."

## 👥 Bu Projeyi Kim Seçer?
- Astronomi/uzay merakı olanlar (çoğunluk!)
- Bilim kurgu sevenler
- "Yeni bir Dünya keşfedebilir miyiz?" sorusu cazip gelenler
- Saf sayısal sınıflandırma sevenler (kategorik öznitelik yok, encoding gerekmiyor)

## 🎯 Beklenen Çıktı (2 saat sonunda)
- Gerçek/yanlış pozitif sınıflandırma modeli
- En önemli ipuçları belirlenmiş (genelde SNR ve transit derinliği)
- Yaşanabilir bölge sayımı yapılmış
- Dünya benzeri gezegen filtrelemesi (genelde 0-3 örnek bulunur — gerçeğe yakın)

## 📊 Beklenen Sayısal Sonuçlar
- **Toplam aday:** ~7300 (gerçek NASA Kepler kataloğu, sadece CONFIRMED + FALSE POSITIVE etiketli olanlar)
- **Gerçek gezegen oranı:** ~%37-38 (gerçekte yanlış pozitifler daha çok)
- **Yaşanabilir bölgedeki gerçek gezegen sayısı:** ~50-55 (gerçek Kepler bulgularıyla uyumlu!)
- **Rastgele Orman doğruluk:** ~%90-92 (gerçek veride güçlü)
- **En önemli öznitelikler (genelde):**
  1. `sinyal_gurultu_orani` (en güçlü)
  2. `transit_derinligi_ppm`
  3. `gezegen_yaricap_dunya`
- **Dünya benzeri sayısı (filtrelenmiş):** ~5-15 aday

## ⚠️ Tipik Takılma Noktaları

**1. "yasanabilir_bolge sütunu sınıflandırmaya da girmiş, doğruluk %100"**
→ `yasanabilir_bolge` öznitelik DEĞİL, ek hedef. `X = uzay[ozellikler]` listesinde olmamalı. Veri sızıntısı uyarısı verin

**2. "Dünya benzeri 0 buldum"**
→ Bu DOĞRU olabilir! Gerçek hayatta Dünya benzeri çok az (Kepler ~50 tane). Bu sentetik veride genelde 5-15 olur. Filtreleri biraz gevşetmelerini söyleyin

**3. "Modelimin doğruluğu %62 — bu kötü mü?"**
→ Bu **baseline** (her şeyi gerçek desek) % 62 zaten. Model bunu geçmeli. Beklenen ~%90-92. Eğer modelin %62 ise bir şey yanlış — `n_estimators=100` olduğundan emin olun

**4. Boolean sütun karışıklığı**
→ `gercek_gezegen` True/False. Bazı öğrenciler `== 1` veya `== 'True'` deneyebilir. Hatırlatın: Pandas boolean'da doğrudan filtreleme yapar (`uzay[uzay['gercek_gezegen']]`)

**5. "SNR neden bu kadar baskın?"**
→ Sentetik veride SNR ana ipucu olarak inşa edildi (gerçek hayatta da öyle). Açıklayın: "Düşük SNR = belirsiz sinyal = muhtemelen gürültü/yanlış pozitif"

## 🎯 Derinleştirme Soruları

1. "Sıcak Jüpiterler kaç tane?" → Yörünge < 10 gün + yarıçap > 8 Dünya
2. "M cüceler etrafındaki gezegenler farklı mı?" → Yıldız sıcaklığı < 4000K filtresi
3. "Eşit-yarıçaplı gezegenleri (1-2 Dünya) ayırırsa, model hangi sıcaklıklarda daha iyi?"
4. "Gerçek hayatta NASA bu modeli kullanırsa, %2 yanlış pozitif verirse 10000 adayda 200 yanlış. Astronomlar 200 adayı manuel inceler. Ne kadar zaman / para kaybeder?"
5. "Modelin tahmin olasılığı 0.4-0.6 olan adayları izole edin — bunlar 'tekrar bakılacak' kümesi. NASA'nın gerçek iş akışı bu!"

## 💭 Etik / Sosyal Tartışma

Uzay araştırmasının etik sorusu şaşırtıcı olabilir:

> "Yaşanabilir bir gezegen bulsak ve oraya 'merhaba' sinyali yollasak, ne riskler var?"

Yönlendirme noktaları:
- **METI tartışması:** Sinyal gönderme = pozisyonumuzu açıklama. Stephen Hawking riskli buldu
- **Bilim yarışı:** Hangi ülke önce keşfederse 'sahip olur' mu? (Cevap: hayır, ama prestij var)
- **Bilimsel yayın:** Algoritma kararı verir, ama insan doğrular. Bu sıralama önemli (otomatik karar tehlikeli)
- **NASA bütçesi:** Vergi paramız uzaya gitsin mi? Sosyal sorunlara gitsin mi? Bu ahlaki bir karar
- **"Yapay Zeka NASA'yı işten çıkardı":** Algoritmalar her geçen yıl daha çok karar veriyor. Astronomların işi azalıyor mu?

## 🎤 Sunum Koçluğu

Uzay sunumları **görsel olarak çok güçlü** olabilir:
- 1 NASA görseli (telif yok, kamuya açık) — Kepler uydusu, ötegezegen sanatçı çizimi
- Karmaşıklık matrisi (gerçek vs yanlış pozitif)
- En önemli 3 öznitelik
- **"Bizim Dünya benzeri listemiz"** tablosu — sınıfa heyecan verir
- Etik kapanış: "Kimse algoritmaya tek başına güvenip 'yeni bir Dünya bulduk' demez. Niye?"

**Bonus kariyer kıvılcımı:** "NASA'da makine öğrenmesi çalışan astronomlar var. Türkiye'de TÜBİTAK Uzay Araştırmaları, TUBİTAK Bilim Kampı — siz de oraya gidebilirsiniz."